In [ ]:
# %pip install google-api-python-client
# %pip install google-auth-httplib2
# %pip install google-auth-oauthlib
# %pip install pandas numpy

In [ ]:
import os
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

SCOPES = [
    "https://www.googleapis.com/auth/googlehealth.activity_and_fitness.readonly",
    "https://www.googleapis.com/auth/googlehealth.location.readonly"
]

def authenticate_google():
    """處理個人 Google 帳號 OAuth 授權認證"""
    creds = None
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file('client_secret.json', SCOPES)
            creds = flow.run_local_server(port=0)
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    return creds

In [ ]:
import requests

def fetch_running_records(access_token, page_token=None):
    url = "https://health.googleapis.com/v4/users/me/dataTypes/exercise/dataPoints"
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Accept": "application/json"
    }

    response = requests.get(url, headers=headers, params={"pageToken": page_token})

    if response.status_code == 200:
        return response.json()
    else:
        print(f"API 請求失敗: HTTP {response.status_code}")
        print(response.text)
        return None



def fetch_running_tcx(access_token, exercise_data_id):
    url = f"https://health.googleapis.com/v4/users/me/dataTypes/exercise/dataPoints/{exercise_data_id}:exportExerciseTcx?alt=media"
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Accept": "application/json"
    }

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        return response.content
    else:
        print(f"API 請求失敗: HTTP {response.status_code}")
        print(response.text)
        return None

In [ ]:
import json

creds = authenticate_google()

exercise_ID = []
page_token = None

while True:
    data = fetch_running_records(creds.token, page_token=page_token)

    for i in data['dataPoints']:

        exerciseType = i['exercise']['exerciseType']
        activeDuration = i['exercise']['activeDuration']

        if exerciseType in ['RUNNING', 'TREADMILL']:
            if int(activeDuration.replace('s', '')) > 60:
                id = i['name'].split('/')[-1]
                exercise_ID.append(id)
                
                OUTPUT_FILENAME = f"{id}.json"

                # 以二進位寫入模式 ('wb') 儲存檔案
                with open(OUTPUT_FILENAME, "w", encoding="utf-8") as f:
                    json.dump(data, f, indent=4, ensure_ascii=False)

    page_token = data.get("nextPageToken")
    if not page_token:
        print("已抓完所有頁面!")
        break

exercise_ID

In [ ]:
for id in exercise_ID:
    runData = fetch_running_tcx(creds.token, id)
    OUTPUT_FILENAME = f"{id}.tcx"

    # 以二進位寫入模式 ('wb') 儲存檔案
    with open(OUTPUT_FILENAME, "wb") as f:
        f.write(runData)
